# 01 - Explore canonical candles

Sanity-check the three canonical timeframes for a single ticker by:

1. Reading rows out of `candles_1m_1s`, `candles_1h_1m`, and `candles_1d_1h`.
2. Checking row count, time-range continuity, and presence of CVD / OHLC.
3. Computing multi-period RSI on each timeframe (`rsi_7`, `rsi_14`, `rsi_28`)
   using the same code path the CLI uses, then plotting price + RSI.

This notebook does not write to the database. To persist the same RSI
series into `features_v1`, use:

```bash
uv run python -m backtest.cli build-features rsi \
    --ticker ES --timeframe 1h_1m --period 7 --period 14 --period 28 \
    --start 2026-01-01 --end 2026-04-01
```

Set `TIMESCALE_DB_URL` before running, pointing at the same Timescale DB
used by `apps/write-node`.

In [ ]:
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd

from backtest.db.candles import read_candles
from backtest.features.builder import compute_feature_series
from backtest.features.registry import default_rsi_specs

TICKER = "ES"
START = datetime(2026, 1, 1, tzinfo=timezone.utc)
END = datetime(2026, 2, 1, tzinfo=timezone.utc)

## 1. Row counts and continuity per timeframe

Quick sanity check across the three canonical layers. For a typical
session-day, expect roughly:

- `1m_1s`: one row per second of open-market time.
- `1h_1m`: one row per minute of open-market time.
- `1d_1h`: one row per hour of open-market time.

If a layer is empty, run the corresponding historical script first.
See `apps/write-node/docs/backfill.md`.

In [ ]:
summary = []
frames: dict[str, pd.DataFrame] = {}
for timeframe in ("1m_1s", "1h_1m", "1d_1h"):
    df = read_candles(ticker=TICKER, timeframe=timeframe, start=START, end=END)
    frames[timeframe] = df
    if df.empty:
        summary.append({"timeframe": timeframe, "rows": 0, "first": None, "last": None})
        continue
    summary.append({
        "timeframe": timeframe,
        "rows": len(df),
        "first": df.index.min().isoformat(),
        "last": df.index.max().isoformat(),
        "close_min": df["close"].min(),
        "close_max": df["close"].max(),
        "cvd_close_first": df["cvd_close"].iloc[0] if "cvd_close" in df.columns else None,
        "cvd_close_last": df["cvd_close"].iloc[-1] if "cvd_close" in df.columns else None,
    })
pd.DataFrame(summary)

## 2. Multi-period RSI on each timeframe

We compute `rsi_7`, `rsi_14`, `rsi_28` independently per canonical
timeframe. The CLI persists exactly these series into `features_v1`.

In [ ]:
specs = default_rsi_specs()

rsi_by_timeframe: dict[str, pd.DataFrame] = {}
for timeframe, df in frames.items():
    if df.empty:
        continue
    columns = {spec.name: compute_feature_series(df, spec) for spec in specs}
    rsi_by_timeframe[timeframe] = pd.DataFrame(columns)

{tf: f.tail(3) for tf, f in rsi_by_timeframe.items()}

## 3. Plot price + RSI(14)

One subplot per timeframe. RSI bounds are 0/100 with 30/70 reference lines.
If the data is empty for any timeframe this cell is a no-op for that layer.

In [ ]:
non_empty = [tf for tf, df in frames.items() if not df.empty]
if not non_empty:
    print("No canonical rows in the requested range. Run the historical backfill first.")
else:
    fig, axes = plt.subplots(
        len(non_empty), 2, figsize=(14, 3 * len(non_empty)), sharex=False
    )
    if len(non_empty) == 1:
        axes = [axes]
    for row, timeframe in enumerate(non_empty):
        df = frames[timeframe]
        rsi = rsi_by_timeframe[timeframe]
        axes[row][0].plot(df.index, df["close"], label="close")
        axes[row][0].set_title(f"{TICKER} {timeframe} close")
        axes[row][0].grid(True, alpha=0.3)
        axes[row][1].plot(rsi.index, rsi["rsi_14"], label="rsi_14")
        axes[row][1].axhline(70, color="red", linestyle=":", alpha=0.5)
        axes[row][1].axhline(30, color="green", linestyle=":", alpha=0.5)
        axes[row][1].set_ylim(0, 100)
        axes[row][1].set_title(f"{TICKER} {timeframe} RSI(14)")
        axes[row][1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()